In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv('data/03_telco_transformed.csv')

# **Predictive Modelling**

## **Introduction**

<small>**In this section, I will be utilising machine learning algorithms to analyse historical data and forecast future outcomes for the Telcom company. In a business context, customer retention is often critical to maintaining profitability. By predicting which customers are at risk of leaving (churning), companies can take proactive measures to improve retention strategies.**

## **1. Context of Data**

<small>This dataset **(03_telco_transformed.csv)** contains records of individual telecommunications customer accounts. It presents a **classic binary classification problem**, where the goal is to predict a distinct two-class outcome: whether a customer will churn or remain with the service. 

**Target variable:** Churn, serves as the primary focal point, using a binary label where 0 represents an active customer and 1 indicates a customer who has left.


**Sample Size:** 7,032 records (customer accounts) across 23 total features (columns).

In [3]:
#Verify the shape of the dataset
print(df.shape)
print("The number of rows in the dataset is: ", df.shape[0])
print("The number of columns in the dataset is: ", df.shape[1])

(7032, 23)
The number of rows in the dataset is:  7032
The number of columns in the dataset is:  23


In [4]:
#Check the column names in the dataset
print(df.columns)

Index(['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService',
       'PaperlessBilling', 'MonthlyCharges', 'Churn',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_Yes', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes',
       'StreamingTV_Yes', 'StreamingMovies_Yes', 'Contract_One year',
       'Contract_Two year'],
      dtype='str')


## **2. The Transformed Data**

<small> The raw dataset has undergone pre-processing and feature transformation to prepare it for machine learning algorithms.

Here is the transformation carried out in the previous sections:

1. **Missing Values Removal:** During the Data Cleaning stage, rows with incomplete profiles were handled, leaving 7,032 complete records. 

2. **No Categorical Strings Remaining:** All features are encoded as integer flags (0 or 1) or continuous numeric values (float64 / int64).

3. **Numeric Encoding:** Categorical text fields (e.g., payment methods, contract types, internet service options) have been converted into binary indicator variables (One-Hot Encoding).

## **3. Selecting the Models to be Tested**

### **A. Logistic Regression**

<small>**In the context of the Telco dataset analyzed earlier, predicting whether a customer will churn (1) or remain (0) is a classic binomial classification task which is why Logistic Regression serves as an effective linear baseline model**

<small> Contrary to linear regression, which predicts continuous values it predicts the probability that an input belongs to a specific class.

Logistic Regression is applied to binary classification where the output can be either of two possible categories such as Yes/No, True/False or 0/1.

The sigmoid function is used to convert inputs into a probability value between 0 and 1.

#### **Assumptions**

<small>

1. Binary OutcomeThe Rule: The target variable must be binary (two categories) for standard binomial logistic regression.

2. Each data point (customer record) must be statistically independent of every other data point.

3. Continuous independent variables (tenure and MonthlyCharges) must have a linear relationship with the log-odds (logit) of the target variable, not directly with the probability itself.

4. Predictor features should not be highly correlated with one another (TotalCharges was dropped earlier due to this)

5. Sufficiently Large Sample Size (Our dataset had 7,032 records which satisfies this assumption)

#### **Mathematical Framework**



Logistic regression models the probability of churn $P(Y = 1 \mid \mathbf{X})$ through three mathematical stages:

1. **The Logit Link (Log-Odds Equation):**
   $$z = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p$$

2. **The Sigmoid / Logistic Function:**
   $$\hat{p} = P(\text{Churn} = 1 \mid \mathbf{X}) = \frac{1}{1 + e^{-z}}$$

3. **The Binary Decision Threshold (Binomial Output):**
   $$\hat{Y} = \begin{cases} 1 & \text{if } \hat{p} \ge 0.5 \quad (\text{Customer Churns}) \\ 0 & \text{if } \hat{p} < 0.5 \quad (\text{Customer Retained}) \end{cases}$$

### **Applying Logistic Regression on the dataset**

##### **Step 1: Isolating the Target Variable (Churn)**

<small> **I will begin by importing the relevant libraries, and then separate the predictors/independant (X) variables from the target variable (y, Churn). This is because machine learning algorithms require a clear distinction between the inputs (features) and the output (target) during training.**

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix, classification_report
)



# Separate features (X) and target (y)
# Dropping Churn from your features to create your clean matrix (X)
X = df.drop(columns=['Churn'])

#Removing the churn by itself for modelling purposes
# Isolating my target variable (y)

y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

#Now X has exactly 22 columns, y is the 1D target array, and is ready for modelling!

Features shape: (7032, 22)
Target distribution:
Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64


##### **Step 2: Split Data into Train and Test Sets**

In [7]:
# Splitting data into 80% train and 20% test
# 'stratify=y' ensures balanced proportions of churn in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"\nTraining target balance:\n{y_train.value_counts(normalize=True)}")
print(f"\nTesting target balance:\n{y_test.value_counts(normalize=True)}")

Training set: 5625 samples
Testing set:  1407 samples
Training features shape: (5625, 22)
Testing features shape: (1407, 22)

Training target balance:
Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Testing target balance:
Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


##### **Step 3: Feature Scaling (Standardization)**

<small> We have two continuous variables: **tenure** and **MonthlyCharges** with much larger scales. Tenure ranges from 1 to 72 months while MonthlyCharges are listed $18 to $118 (Refer to chart plotted in 03_data_transformation.ipynb). I will apply scaling which should ensure stable gradient convergence.

In [9]:
# Initialising and fitting the scaler
scaler = StandardScaler()

# Transforming  features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Continuous features have been scaled successfully!")

Continuous features have been scaled successfully!


##### **Step 4: Model Instantiation and Training**

In [ ]:
#Here, I will be instantiating LogisticRegression and fitting it using the scaled training features.
#We use class_weight='balanced' to handle the 73/27 data split perfectly.
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Fitting model on training data
log_reg.fit(X_train_scaled, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.
